In [1]:
import sys

sys.path.insert(0, "..")  # repo root, where column_analysis.py lives
import numpy as np
from column_analysis import (
    load_data,
    column_profile,
    print_label_rates,
    find_exact_duplicates,
    find_correlated_pairs,
    find_determined_columns,
    choose_determined_drops,
    leakage_scan,
    save_decisions,
    summarize_decisions,
)

DATA_PATH = "../data/"  # x_train.csv, y_train.csv, x_test.csv
CACHE = "../data/cache.npz"
DECISIONS_PATH = "../decisions/column_decisions.csv"

x_train, x_test, y_train, train_ids, test_ids, names = load_data(DATA_PATH, CACHE)
y01 = (y_train == 1).astype(float)
profile = column_profile(x_train, x_test, y01, names)
P = {p["name"]: p for p in profile}
print(
    "train",
    x_train.shape,
    "| test",
    x_test.shape,
    "| positive rate",
    round(y01.mean(), 4),
)


train (328135, 321) | test (109379, 321) | positive rate 0.0883


In [2]:
from helpers import load_csv_data
from column_analysis import load_decisions
from preprocessing import fit_preprocess, transform, print_report

decisions = load_decisions("../decisions/column_decisions.csv")

params = fit_preprocess(x_train, names, decisions)   # learns from TRAIN only
x_train_p = transform(x_train, params)
x_test_p = transform(x_test, params)                 # same params, no refitting

print_report(params)           # optional: what happened to each column
params["feature_names"]        # names of the output columns

SUMMARY: one-hot: 188, dropped: 54, continuous: 79 | final number of features: 1279

--- dropped (54) ---
  FMONTH                                                                  | interview or survey weighting variable, not about the person (cell 3)
  IDATE                                                                   | interview or survey weighting variable, not about the person (cell 3)
  IMONTH                                                                  | interview or survey weighting variable, not about the person (cell 3)
  IDAY                                                                    | interview or survey weighting variable, not about the person (cell 3)
  IYEAR                                                                   | interview or survey weighting variable, not about the person (cell 3)
  DISPCODE                                                                | interview or survey weighting variable, not about the person (cell 3)
  SEQNO           

['_STATE=1',
 '_STATE=2',
 '_STATE=4',
 '_STATE=5',
 '_STATE=6',
 '_STATE=8',
 '_STATE=9',
 '_STATE=10',
 '_STATE=11',
 '_STATE=12',
 '_STATE=13',
 '_STATE=15',
 '_STATE=16',
 '_STATE=17',
 '_STATE=18',
 '_STATE=19',
 '_STATE=20',
 '_STATE=21',
 '_STATE=22',
 '_STATE=23',
 '_STATE=24',
 '_STATE=25',
 '_STATE=26',
 '_STATE=27',
 '_STATE=28',
 '_STATE=29',
 '_STATE=30',
 '_STATE=31',
 '_STATE=32',
 '_STATE=33',
 '_STATE=34',
 '_STATE=35',
 '_STATE=36',
 '_STATE=37',
 '_STATE=38',
 '_STATE=39',
 '_STATE=40',
 '_STATE=41',
 '_STATE=42',
 '_STATE=44',
 '_STATE=45',
 '_STATE=46',
 '_STATE=47',
 '_STATE=48',
 '_STATE=49',
 '_STATE=50',
 '_STATE=51',
 '_STATE=53',
 '_STATE=54',
 '_STATE=55',
 '_STATE=56',
 '_STATE=66',
 '_STATE=72',
 'PVTRESD1=1',
 'PVTRESD1=missing',
 'STATERES=1',
 'STATERES=missing',
 'CELLFON3=1',
 'CELLFON3=2',
 'CELLFON3=missing',
 'LADULT=missing',
 'NUMADULT_missing',
 'NUMADULT',
 'NUMMEN_missing',
 'NUMMEN',
 'NUMWOMEN_missing',
 'NUMWOMEN',
 'CADULT=1',
 'CADULT=2',

In [3]:
from extra_implementations import grid_search_KNN

# 10-fold CV on the training set only; PCA is refit inside every fold
gs = grid_search_KNN(
    n_components=(10, 20, 30, 50, 70, 100),
    thresholds=(0.0833, 0.1, 0.2, 0.3, 0.5),
    n_neighbors=(5, 10, 15, 20),
    n_splits=5,
    batch_size=256,
).fit(x_train, y01, names, decisions)

print('best params:', gs.best_params_)
print('best CV F1 :', round(gs.best_score_, 4))
for r in gs.results_[:10]:
    print(r)

fold 5/5 | evaluations 600/600 (100%) | elapsed 219.7 min, ETA 0.0 minn
best params: {'n_components': 10, 'n_neighbors': 20, 'threshold': 0.1}
best CV F1 : 0.3281
{'n_components': 10, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.32805983128455096), 'std_score': np.float64(0.0014955604885478764)}
{'n_components': 20, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.3268506212843331), 'std_score': np.float64(0.0009999200923543347)}
{'n_components': 100, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.32651254687332887), 'std_score': np.float64(0.0018383141934728404)}
{'n_components': 50, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.3264622379702681), 'std_score': np.float64(0.002907146153496055)}
{'n_components': 30, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.3256992978436861), 'std_score': np.float64(0.002041437060718271)}
{'n_components': 70, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.f

In [4]:
from extra_implementations import grid_search_KNN

gs = grid_search_KNN(
    n_components=(10, 15, 20),
    thresholds=(0.0883, 0.09, 0.1),
    n_neighbors=(20, 25, 30),
    n_splits=5,
    batch_size=256,
).fit(x_train, y01, names, decisions)

print('best params:', gs.best_params_)
print('best CV F1 :', round(gs.best_score_, 4))
for r in gs.results_[:10]:
    print(r)

fold 5/5 | evaluations 135/135 (100%) | elapsed 77.8 min, ETA 0.0 min
best params: {'n_components': 15, 'n_neighbors': 30, 'threshold': 0.1}
best CV F1 : 0.334
{'n_components': 15, 'n_neighbors': 30, 'threshold': 0.1, 'mean_score': np.float64(0.333954612605939), 'std_score': np.float64(0.001489134321540846)}
{'n_components': 15, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.33193750458844196), 'std_score': np.float64(0.0011926611010430613)}
{'n_components': 10, 'n_neighbors': 30, 'threshold': 0.1, 'mean_score': np.float64(0.33100209172999795), 'std_score': np.float64(0.0024146618501878553)}
{'n_components': 20, 'n_neighbors': 30, 'threshold': 0.1, 'mean_score': np.float64(0.3303529898791871), 'std_score': np.float64(0.0010951932121472153)}
{'n_components': 10, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.float64(0.32805983128455096), 'std_score': np.float64(0.0014955604885478764)}
{'n_components': 20, 'n_neighbors': 20, 'threshold': 0.1, 'mean_score': np.floa